# Voice-conversion backend on Kaggle (free GPU)
**Settings (right sidebar):** Accelerator = **GPU T4/P100**, Internet = **On**.

Run the cells top to bottom. The last cell keeps running — leave it running (and this tab open) while people use the website.

## 1. Your tokens (edit this cell)

In [ ]:
HF_TOKEN     = "hf_PASTE_HUGGINGFACE_READ_TOKEN"
NGROK_TOKEN  = "PASTE_NGROK_AUTHTOKEN"
NGROK_DOMAIN = ""   # e.g. "your-name.ngrok-free.dev" (your free static domain). Leave "" for a random URL.


## 2. Install Applio (RVC)

In [ ]:
!pip install -q uv
!git clone --depth 1 https://github.com/IAHispano/Applio.git /kaggle/working/Applio
!apt install -y -qq portaudio19-dev > /dev/null 2>&1
%cd /kaggle/working/Applio
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system
!python core.py prerequisites --models --exe --pretraineds-hifigan > /dev/null 2>&1
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok huggingface_hub
import torch; print("cuda:", torch.cuda.is_available())

## 3. Backend files

In [ ]:
%%writefile /kaggle/working/backend/prep_models.py
"""Download the 4 RVC speaker zips from Hugging Face -> MODELS_DIR/speaker_N/model.pth (+ .index)."""
import os, re, glob, shutil, zipfile
from huggingface_hub import hf_hub_download

REPO = "Speech-itu/Urdu_Speakers_Models_BSAI"
OUT = os.getenv("MODELS_DIR", "/kaggle/working/models/rvc")
TOKEN = os.environ["HF_TOKEN"]

SPEAKERS = [int(x) for x in os.getenv("SPEAKERS", "1,2,3,4").split(",")]
for n in SPEAKERS:
    z = hf_hub_download(REPO, f"rvc_speaker_{n}.zip", token=TOKEN)
    d = f"{OUT}/speaker_{n}"; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    names = zipfile.ZipFile(z).namelist()

    if any(x.endswith("data.pkl") for x in names) and not any(x.endswith(".pth") for x in names):
        shutil.copy(z, f"{d}/model.pth")            # the "zip" is itself a torch .pth file
    else:
        tmp = f"{d}/_x"; zipfile.ZipFile(z).extractall(tmp)
        def ok(p): return "__MACOSX" not in p and not os.path.basename(p).startswith(("D_", "G_", "._"))
        def epoch(p):                                   # e.g. midage_150e_6450s.pth -> 150
            m = re.search(r"(\d+)e_", os.path.basename(p)); return int(m.group(1)) if m else 0
        pths = [p for p in glob.glob(tmp + "/**/*.pth", recursive=True) if ok(p)]
        print(f"speaker_{n} candidates:", sorted(os.path.basename(p) for p in pths))
        if pths:
            best = max(pths, key=epoch)                 # highest epoch = latest checkpoint
            print(f"speaker_{n} using:", os.path.basename(best))
            shutil.copy(best, f"{d}/model.pth")
        else:                                        # extracted torch folder -> repack as .pth
            pk = glob.glob(tmp + "/**/data.pkl", recursive=True)[0]
            top = os.path.dirname(pk)
            with zipfile.ZipFile(f"{d}/model.pth", "w", zipfile.ZIP_STORED) as zz:
                for root, _, files in os.walk(top):
                    for fn in files:
                        full = os.path.join(root, fn); zz.write(full, os.path.relpath(full, os.path.dirname(top)))
        for ix in glob.glob(tmp + "/**/*.index", recursive=True):
            if "__MACOSX" not in ix and not os.path.basename(ix).startswith("._"):
                shutil.copy(ix, d)
        shutil.rmtree(tmp)
    print(f"speaker_{n}:", os.listdir(d))


In [ ]:
%%writefile /kaggle/working/backend/main.py
"""Voice-conversion API (FastAPI + Applio/RVC). One GPU worker, async jobs."""
import os, uuid, time, shutil, subprocess, glob
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

APPLIO = os.getenv("APPLIO_DIR", "/kaggle/working/Applio")
PYTHON = os.getenv("APPLIO_PYTHON", "python")
SEEDVC = os.getenv("SEEDVC_DIR", "/kaggle/working/seed-vc")
SEEDVC_PYTHON = os.getenv("SEEDVC_PYTHON", PYTHON)
ENABLED = {int(x) for x in os.getenv("ENABLED_SPEAKERS", "1,2,3,4").split(",")}
MODELS = Path(os.getenv("MODELS_DIR", "/kaggle/working/models/rvc"))
WORK = Path(os.getenv("WORK_DIR", "/tmp/vc_jobs")); WORK.mkdir(parents=True, exist_ok=True)
ORIGINS = os.getenv(
    "ALLOWED_ORIGINS",
    "https://voice-conversion-vc-fyp-frontend.vercel.app,http://localhost:5500,http://127.0.0.1:5500,http://localhost:3000",
).split(",")

# (gender, age) -> speaker number.  1=old man, 2=old woman, 3=mid-age man, 4=mid-age woman
SPEAKERS = {("male", "old"): 1, ("female", "old"): 2, ("male", "adult"): 3, ("female", "adult"): 4}
# default pitch shift (semitones) per speaker; tune these by ear
PITCH = {1: 0, 2: 0, 3: 0, 4: 0}

MAX_UPLOAD = 20 * 1024 * 1024
JOBS: dict = {}
pool = ThreadPoolExecutor(max_workers=1)   # one conversion at a time on the GPU

app = FastAPI(title="Voice Conversion API")
app.add_middleware(CORSMiddleware, allow_origins=ORIGINS, allow_origin_regex=r"https://.*\.vercel\.app",
                   allow_methods=["*"], allow_headers=["*"])


def model_files(n: int):
    d = MODELS / f"speaker_{n}"
    idx = next(iter(glob.glob(str(d / "*.index"))), None)
    return d / "model.pth", idx


def seedvc_files(n: int):
    """(checkpoint, config, reference wav) for the fine-tuned Seed-VC stage, or None if not installed."""
    d = MODELS / f"speaker_{n}" / "seedvc"
    t = (d / "ckpt.pth", d / "config.yml", d / "ref.wav")
    return t if all(p.exists() for p in t) else None


def run_job(job_id: str, speaker: int, pitch: int):
    job = JOBS[job_id]; d = WORK / job_id
    try:
        job["status"] = "processing"
        src, wav, out = d / "input", d / "in.wav", d / "out.wav"
        r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src), "-t", "90",
                            "-vn", "-ac", "1", "-ar", "40000", str(wav)], capture_output=True, text=True)
        if r.returncode: raise RuntimeError("Could not read the audio: " + r.stderr[-300:])

        pth, idx = model_files(speaker)
        cmd = [PYTHON, "core.py", "infer", "--input-path", str(wav), "--output-path", str(out),
               "--pth-path", str(pth), "--index-rate", "0.5" if idx else "0", "--pitch", str(pitch),
               "--f0-method", "rmvpe", "--embedder-model", "contentvec", "--protect", "0.33",
               "--export-format", "WAV"]
        if idx: cmd += ["--index-path", idx]
        job["stage"] = "voice"                       # stage 1: RVC
        r = subprocess.run(cmd, cwd=APPLIO, capture_output=True, text=True, timeout=600)
        if not out.exists(): raise RuntimeError("Conversion failed: " + (r.stdout + r.stderr)[-500:])
        job["file"] = str(out)

        sv = seedvc_files(speaker)
        if sv:                                       # stage 2: fine-tuned Seed-VC style transfer
            job["stage"] = "style"
            sdir = d / "sv"; sdir.mkdir(exist_ok=True)
            ck, cfg, ref = sv
            r = subprocess.run([SEEDVC_PYTHON, "inference.py", "--source", str(out), "--target", str(ref),
                                "--output", str(sdir), "--checkpoint", str(ck), "--config", str(cfg),
                                "--diffusion-steps", "50", "--length-adjust", "1.0",
                                "--inference-cfg-rate", "0.7", "--f0-condition", "False"],
                               cwd=SEEDVC, capture_output=True, text=True, timeout=900)
            wavs = sorted(glob.glob(str(sdir / "*.wav")))
            if not wavs: raise RuntimeError("Style transfer failed: " + (r.stdout + r.stderr)[-500:])
            job["file"] = wavs[0]
        job["status"] = "done"
    except Exception as e:
        job["status"], job["error"] = "error", str(e)


def cleanup():
    now = time.time()
    for jid in [j for j, v in JOBS.items() if now - v["t"] > 3600]:
        JOBS.pop(jid, None); shutil.rmtree(WORK / jid, ignore_errors=True)


@app.get("/health")
def health():
    return {"ok": True, "enabled": sorted(ENABLED),
            "rvc": {n: model_files(n)[0].exists() for n in (1, 2, 3, 4)},
            "seedvc": {n: seedvc_files(n) is not None for n in (1, 2, 3, 4)}}


@app.post("/convert")
async def convert(audio: UploadFile = File(...), gender: str = Form(...), age: str = Form(...),
                  pitch: int | None = Form(None)):
    cleanup()
    speaker = SPEAKERS.get((gender.lower(), age.lower()))
    if not speaker: raise HTTPException(400, "Unsupported voice. Choose male/female and adult/old.")
    if speaker not in ENABLED: raise HTTPException(400, "This voice is not available yet.")
    if not model_files(speaker)[0].exists(): raise HTTPException(500, f"Model for speaker {speaker} is not installed.")
    data = await audio.read()
    if not data: raise HTTPException(400, "Empty audio.")
    if len(data) > MAX_UPLOAD: raise HTTPException(413, "Audio too large (max 20 MB).")

    job_id = uuid.uuid4().hex
    (WORK / job_id).mkdir(parents=True)
    (WORK / job_id / "input").write_bytes(data)
    p = PITCH[speaker] if pitch is None else max(-24, min(24, pitch))
    JOBS[job_id] = {"status": "queued", "t": time.time()}
    pool.submit(run_job, job_id, speaker, p)
    return {"job_id": job_id}


@app.get("/jobs/{job_id}")
def job_status(job_id: str):
    j = JOBS.get(job_id)
    if not j: raise HTTPException(404, "Unknown job.")
    return {"status": j["status"], "stage": j.get("stage"), "error": j.get("error")}


@app.get("/jobs/{job_id}/audio")
def job_audio(job_id: str):
    j = JOBS.get(job_id)
    if not j or j["status"] != "done": raise HTTPException(404, "Not ready.")
    return FileResponse(j["file"], media_type="audio/wav", filename="converted.wav")


## 4. Download the 4 speaker models

In [ ]:
import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SPEAKERS"] = "1"   # only speaker 1 (old man) for now
os.environ["MODELS_DIR"] = "/kaggle/working/models/rvc"
!python /kaggle/working/backend/prep_models.py
!ls -lh /kaggle/working/models/rvc/*

## 4b. Seed-VC (style-transfer stage)

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Plachtaa/seed-vc.git
%cd /kaggle/working/seed-vc
!grep -viE "^(torch|torchvision|torchaudio)([=<>~ ]|$)" requirements.txt > req_notorch.txt
!pip install -q -r req_notorch.txt 2>&1 | tail -5
!uv pip install -q --system librosa munch einops descript-audio-codec pydub transformers soundfile hydra-core pyyaml "huggingface-hub>=0.28.1" scipy accelerate resemblyzer jiwer

# bigvgan / huggingface_hub compatibility patch
import re, glob, shutil
p = glob.glob("/kaggle/working/seed-vc/modules/**/bigvgan.py", recursive=True)[0]
shutil.copy(p, p + ".bak")
s = open(p).read()
s = re.sub(r"proxies:\s*Optional\[Dict\],", "proxies: Optional[Dict] = None,", s)
s = re.sub(r"resume_download:\s*bool,", "resume_download: bool = False,", s)
s = re.sub(r"^\s*proxies\s*=\s*proxies,\s*\n", "", s, flags=re.M)
s = re.sub(r"^\s*resume_download\s*=\s*resume_download,\s*\n", "", s, flags=re.M)
open(p, "w").write(s); print("patched")

In [ ]:
%%writefile /kaggle/working/backend/prep_seedvc.py
"""Find each speaker's fine-tuned Seed-VC files in the HF repo and lay them out as
MODELS_DIR/speaker_N/seedvc/{ckpt.pth,config.yml,ref.wav}. Works with zips or loose files, any folder name
as long as the path mentions 'seed' and 'speaker_N'."""
import os, re, glob, shutil, zipfile
from huggingface_hub import hf_hub_download, list_repo_files

REPO = os.getenv("SEEDVC_REPO", "Speech-itu/Urdu_Speakers_Models_BSAI")
OUT = os.getenv("MODELS_DIR", "/kaggle/working/models/rvc")
TOKEN = os.environ["HF_TOKEN"]
SPEAKERS = [int(x) for x in os.getenv("SPEAKERS", "1,2,3,4").split(",")]
EXT = (".zip", ".pth", ".yml", ".yaml", ".wav")

def clean(ps): return [p for p in ps if "__MACOSX" not in p and not os.path.basename(p).startswith("._")]
def step(p):
    m = re.search(r"step_(\d+)", os.path.basename(p)); return int(m.group(1)) if m else 0
def of_speaker(f, n): return re.search(rf"speaker[_\- ]?{n}(?!\d)", f.lower()) is not None

files = list_repo_files(REPO, token=TOKEN)
for n in SPEAKERS:
    d = f"{OUT}/speaker_{n}/seedvc"; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    tmp = f"{d}/_x"; os.makedirs(tmp)
    cand = [f for f in files if "seed" in f.lower() and of_speaker(f, n) and f.lower().endswith(EXT)
            and "rvc" not in f.lower()]
    loose = [f for f in cand if not f.lower().endswith(".zip")]
    use = loose if any(f.lower().endswith(".pth") for f in loose) else cand   # prefer the folder, ignore old zips
    print(f"speaker_{n}: using repo files:", use)
    for f in use:
        local = hf_hub_download(REPO, f, token=TOKEN)
        if f.lower().endswith(".zip"): zipfile.ZipFile(local).extractall(f"{tmp}/{os.path.basename(f)}_")
        else:
            os.makedirs(f"{tmp}/{os.path.dirname(f)}", exist_ok=True); shutil.copy(local, f"{tmp}/{f}")

    pths = clean(glob.glob(tmp + "/**/*.pth", recursive=True))
    ft = [p for p in pths if os.path.basename(p).startswith("ft_model")]
    ckpt = ft[0] if ft else (max(pths, key=step) if pths else None)
    ymls = clean(glob.glob(tmp + "/**/*.yml", recursive=True) + glob.glob(tmp + "/**/*.yaml", recursive=True))
    own = [y for y in ymls if "/configs/" not in y]                              # saved next to the weights
    stock = [y for y in ymls if "whisper_small_wavenet" in os.path.basename(y)]  # preset used for training
    cfg = (own or stock or [None])[0]
    wavs = clean(glob.glob(tmp + "/**/*.wav", recursive=True))
    if not wavs:   # no reference clip in the folder -> take one from the old code zip's reference/ folder
        for f in [c for c in cand if c.lower().endswith(".zip")]:
            zf = zipfile.ZipFile(hf_hub_download(REPO, f, token=TOKEN))
            for name in zf.namelist():
                if name.lower().endswith(".wav") and "reference" in name.lower() and "__MACOSX" not in name:
                    zf.extract(name, f"{tmp}/_ref")
        wavs = clean(glob.glob(tmp + "/_ref/**/*.wav", recursive=True))
    wavs = sorted(wavs, key=lambda w: ("ref" not in w.lower(), w))
    print(f"speaker_{n}: ckpt={ckpt and os.path.relpath(ckpt, tmp)} | cfg={cfg and os.path.relpath(cfg, tmp)} | ref={wavs and os.path.relpath(wavs[0], tmp)}")
    if not (ckpt and cfg and wavs):
        print(f"speaker_{n}: INCOMPLETE -> RVC only. Repo files mentioning 'seed':", [f for f in files if "seed" in f.lower()])
        shutil.rmtree(d); continue
    shutil.copy(ckpt, f"{d}/ckpt.pth"); shutil.copy(cfg, f"{d}/config.yml"); shutil.copy(wavs[0], f"{d}/ref.wav")
    shutil.rmtree(tmp, ignore_errors=True)


In [ ]:
!python /kaggle/working/backend/prep_seedvc.py
!ls -lh /kaggle/working/models/rvc/*/seedvc 2>/dev/null

## 5. Start the API + public https URL (leave this cell running)

In [ ]:
import os, subprocess, time, urllib.request, sys
from pyngrok import ngrok

env = dict(os.environ, APPLIO_DIR="/kaggle/working/Applio", APPLIO_PYTHON=sys.executable,
           MODELS_DIR="/kaggle/working/models/rvc", SEEDVC_DIR="/kaggle/working/seed-vc", ENABLED_SPEAKERS="1")
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
                          cwd="/kaggle/working/backend", env=env)
time.sleep(6)
print("health:", urllib.request.urlopen("http://localhost:8000/health").read().decode())

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(8000)
print("\n>>> API_URL for frontend/config.js:", tunnel.public_url, "\n")

while True:           # keeps the session alive
    time.sleep(60)
